In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

The goal of this project is to analyze an e-commerce store's sales and customers using SQL to find revenue drivers and top customer segments

In [2]:
from google.cloud import bigquery
client = bigquery.Client()

dataset_ref = client.dataset("thelook_ecommerce", project="bigquery-public-data")
dataset = client.get_dataset(dataset_ref)

tables = list(client.list_tables(dataset))
for t in tables:
    print(t.table_id)

safe_config = bigquery.QueryJobConfig(
    maximum_bytes_billed = 10 * 10**9   # 10 GB safety cap
)

Using Kaggle's public dataset BigQuery integration.
distribution_centers
events
inventory_items
order_items
orders
products
thelook_ecommerce-table
users


Here we are setting up for the project, including a list of tables and a safe config so we don't accidentally do a query that is too large

In [3]:
key_tables = [
    'users',
    'orders',
    'order_items',
    'products'
]

previews = {}

for table in key_tables:
    table_ref = dataset_ref.table(table)
    previews[table] = client.list_rows(table_ref, max_results = 5).to_dataframe()

this creates a dictionary with previews of the key tables

In [4]:
for table, df in previews.items():
    print(f"{table.upper()}")
    display(df)

USERS


,id,first_name,last_name,email,age,gender,state,street_address,postal_code,city,country,latitude,longitude,traffic_source,created_at,user_geom
0,17258,Monica,Miller,monicamiller@example.com,39,F,Acre,46663 Tricia Tunnel Apt. 978,69980-000,null,Brasil,-8.065346,-72.870949,Search,2023-12-27 09:58:00+00:00,POINT(-72.87094866 -8.065346116)
1,62423,Melvin,Hernandez,melvinhernandez@example.org,33,M,Acre,560 Justin Glen Apt. 393,69980-000,null,Brasil,-8.065346,-72.870949,Search,2020-01-18 06:44:00+00:00,POINT(-72.87094866 -8.065346116)
2,5220,Whitney,Harmon,whitneyharmon@example.com,20,F,Acre,66865 Morgan Burgs,69980-000,null,Brasil,-8.065346,-72.870949,Search,2024-11-13 06:52:00+00:00,POINT(-72.87094866 -8.065346116)
3,1429,Matthew,Roberts,matthewroberts@example.com,60,M,Acre,428 Rosales Tunnel,69980-000,null,Brasil,-8.065346,-72.870949,Search,2022-04-21 00:13:00+00:00,POINT(-72.87094866 -8.065346116)
4,71326,Rachel,Quinn,rachelquinn@example.org,41,F,Acre,792 Andre Locks,69980-000,null,Brasil,-8.065346,-72.870949,Search,2021-10-01 03:52:00+00:00,POINT(-72.87094866 -8.065346116)


ORDERS


,order_id,user_id,status,gender,created_at,returned_at,shipped_at,delivered_at,num_of_item
0,16,13,Cancelled,F,2025-01-18 10:09:00+00:00,NaT,NaT,NaT,1
1,27,24,Cancelled,F,2025-07-18 09:39:00+00:00,NaT,NaT,NaT,1
2,39,37,Cancelled,F,2025-02-11 08:21:00+00:00,NaT,NaT,NaT,1
3,43,42,Cancelled,F,2024-04-16 02:55:00+00:00,NaT,NaT,NaT,1
4,68,65,Cancelled,F,2025-10-28 14:58:00+00:00,NaT,NaT,NaT,1


ORDER_ITEMS


,id,order_id,user_id,product_id,inventory_item_id,status,created_at,shipped_at,delivered_at,returned_at,sale_price
0,15280,10462,8336,14235,41264,Cancelled,2020-12-23 05:30:39+00:00,NaT,NaT,NaT,0.02
1,76388,52739,42049,14235,205977,Complete,2020-05-04 11:55:17+00:00,2020-05-07 01:28:00+00:00,2020-05-07 19:25:00+00:00,NaT,0.02
2,178038,122609,97987,14235,480749,Complete,2024-08-20 04:29:18+00:00,2024-08-20 08:05:00+00:00,2024-08-23 21:58:00+00:00,NaT,0.02
3,149765,103240,82584,14235,404313,Processing,2024-02-23 16:28:31+00:00,NaT,NaT,NaT,0.02
4,152499,105067,84052,14235,411693,Processing,2022-11-24 05:29:50+00:00,NaT,NaT,NaT,0.02


PRODUCTS


,id,cost,category,name,brand,retail_price,department,sku,distribution_center_id
0,13842,2.51875,Accessories,Low Profile Dyed Cotton Twill Cap - Navy W39S55D,MG,6.25,Women,EBD58B8A3F1D72F4206201DA62FB1204,1
1,13928,2.33835,Accessories,Low Profile Dyed Cotton Twill Cap - Putty W39S55D,MG,5.95,Women,2EAC42424D12436BDD6A5B8A88480CC3,1
2,14115,4.87956,Accessories,Enzyme Regular Solid Army Caps-Black W35S45D,MG,10.99,Women,EE364229B2791D1EF9355708EFF0BA34,1
3,14157,4.64877,Accessories,Enzyme Regular Solid Army Caps-Olive W35S45D (...,MG,10.99,Women,00BD13095D06C20B11A2993CA419D16B,1
4,14273,6.50793,Accessories,Washed Canvas Ivy Cap - Black W11S64C,MG,15.99,Women,F531DC20FDE20B7ADF3A73F52B71D0AF,1


this section provides previews while being efficient with queries

In [5]:
monthly_revenue_query = """
    SELECT
        EXTRACT(YEAR FROM created_at) AS year,
        EXTRACT(MONTH FROM created_at) AS month,
        SUM(sale_price) AS revenue
    FROM
        `bigquery-public-data.thelook_ecommerce.order_items` as oi
    GROUP BY
        year, month
    ORDER BY
        year, month ASC
"""

monthly_revenue = client.query(monthly_revenue_query, job_config = safe_config).to_dataframe()

monthly_profit_query = """
    SELECT
        EXTRACT(YEAR FROM oi.created_at) AS year,
        EXTRACT(MONTH FROM oi.created_at) AS month,
        SUM(oi.sale_price) - SUM(p.cost) AS profit
    FROM
        `bigquery-public-data.thelook_ecommerce.order_items` AS oi
    JOIN 
        `bigquery-public-data.thelook_ecommerce.products` as p
        ON oi.product_id = p.id
    GROUP BY
        year, month
    ORDER BY
        year, month

"""


monthly_profit = client.query(monthly_profit_query, job_config = safe_config).to_dataframe()

/usr/local/lib/python3.11/dist-packages/google/cloud/bigquery/table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [6]:
display(monthly_revenue)
display(monthly_profit)


,year,month,revenue
0,2019,1,509.370004
1,2019,2,1480.500002
2,2019,3,2968.980007
3,2019,4,4325.840023
4,2019,5,8034.469988
...,...,...,...
79,2025,8,413608.060419
80,2025,9,433991.140526
81,2025,10,506872.110445
82,2025,11,650801.690650


,year,month,profit
0,2019,1,263.434541
1,2019,2,749.311271
2,2019,3,1516.286374
3,2019,4,2197.501941
4,2019,5,4136.695734
...,...,...,...
79,2025,8,214461.680373
80,2025,9,225205.579782
81,2025,10,262507.134573
82,2025,11,338608.777418


In [7]:
top_product_query = """
    SELECT
        oi.product_id,
        p.name,
        p.category,
        SUM(oi.sale_price) AS revenue,
    SUM(p.cost) AS cost,
    SUM(oi.sale_price) - SUM(p.cost) AS profit

    FROM
        `bigquery-public-data.thelook_ecommerce.order_items` as oi
    JOIN
        `bigquery-public-data.thelook_ecommerce.products` as p
        ON oi.product_id = p.id
    WHERE
        oi.status = 'Complete'
    GROUP BY
        p.name,
        oi.product_id,
        p.category
    ORDER BY
        profit DESC
"""

top_products = client.query(top_product_query, job_config = safe_config).to_dataframe()

/usr/local/lib/python3.11/dist-packages/google/cloud/bigquery/table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [8]:
display(top_products)

,product_id,name,category,revenue,cost,profit
0,24447,Darla,Outerwear & Coats,5994.00,2427.570007,3566.429993
1,23654,The North Face Apex Bionic Soft Shell Jacket -...,Outerwear & Coats,4515.00,1815.030012,2699.969988
2,23646,Diesel Men's Lophophora Leather Jacket,Outerwear & Coats,4490.00,2042.950005,2447.049995
3,23951,The North Face Nuptse 2 Jacket Deep Water Blue...,Outerwear & Coats,3612.00,1470.084005,2141.915995
4,2559,NIKE WOMEN'S PRO COMPRESSION SPORTS BRA *Outst...,Active,3612.00,1614.564009,1997.435991
...,...,...,...,...,...,...
22957,15395,Retractable Colorful Rhinestone Lanyards with ...,Plus,2.67,1.415100,1.254900
22958,14298,Classic Tear Drop Mirror Lens Aviator Sunglasses,Accessories,1.72,0.645000,1.075000
22959,3049,Pink Ribbon Breast Cancer Awareness Knee High ...,Active,1.95,0.916500,1.033500
22960,14159,Set of 2 - Replacement Insert For Checkbook Wa...,Accessories,0.49,0.177380,0.312620


In [11]:
total_customer_spending_query = """
    SELECT
        oi.user_id,
        SUM(oi.sale_price) as total_spending,
        COUNT(*) AS total_items_purchased,
        u.gender,
        u.age,
        u.state
    FROM
        `bigquery-public-data.thelook_ecommerce.users` as u
    JOIN
        `bigquery-public-data.thelook_ecommerce.order_items` as oi
        ON u.id = oi.user_id
    WHERE
        oi.status = 'Complete'
    GROUP BY
        oi.user_id,
        u.gender,
        u.age,
        u.state
    ORDER BY
        total_spending DESC
        
"""

total_customer_spending = client.query(total_customer_spending_query, job_config = safe_config).to_dataframe()

/usr/local/lib/python3.11/dist-packages/google/cloud/bigquery/table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [12]:
display(total_customer_spending)

,user_id,total_spending,total_items_purchased,gender,age,state
0,95880,1257.059992,6,M,14,Bretagne
1,21813,1208.000000,2,M,25,Castilla y León
2,83422,1182.940006,4,M,40,Florida
3,79173,1146.849997,4,M,68,Maranhão
4,4509,1134.930002,4,M,62,Amazonas
...,...,...,...,...,...,...
27542,63132,1.510000,1,F,58,Arizona
27543,17258,1.510000,1,F,39,Acre
27544,26857,1.500000,1,F,68,California
27545,49754,1.500000,1,M,62,Wallonia


In [13]:
monthly_revenue.to_csv("monthly_revenue.csv", index=False)
monthly_profit.to_csv("monthly_profit.csv", index=False)
top_products.to_csv("top_products.csv", index=False)
total_customer_spending.to_csv("total_customer_spending.csv", index=False)
